<a href="https://colab.research.google.com/github/ELZAIM1/CNN/blob/main/CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch

In [2]:
from torchvision import transforms , datasets

preprocessing = transforms.Compose([
    transforms.ToTensor()
])

train = datasets.MNIST(root='./data', train=True, download=True, transform=preprocessing)
test = datasets.MNIST(root='./data', train=False, download=True, transform=preprocessing)

100%|██████████| 9.91M/9.91M [00:00<00:00, 46.0MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.08MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 9.57MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 10.3MB/s]


In [3]:
print(len(train))  #60,000
print(len(test))   #10,000

60000
10000


In [4]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train, shuffle=True, batch_size=500, num_workers=4 )

test_loader = DataLoader(test, batch_size=500, num_workers=4 )

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [5]:
images, labels = next(iter(train_loader))

print ("images: ", images.size())
print("labels: ", labels.size())


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


images:  torch.Size([500, 1, 28, 28])
labels:  torch.Size([500])


In [6]:
import torch.nn as nn
import torch.nn.functional as F

class CustomCNN(nn.Module):
    def __init__(self, num_classes= 10):
        super(CustomCNN, self).__init__()
        # (1 * 28 * 28 )
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1 )

        # (16 * 14 * 14 )
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1 )
        # (32 * 7 * 7 )
        self.pool = nn.MaxPool2d(2,2)

        self.dropout = nn.Dropout(0.4)

        self.fc3 = nn.Linear(32 * 7 * 7 , 128)

        self.fc4 = nn.Linear(128, num_classes)

    def forward(self, x):

        # First Layer CONV1

        x = self.conv1(x)
        x = F.relu(x)
        x = self.pool(x)

        # second Layer CONV2

        x = self.conv2(x)
        x = F.relu(x)
        x = self.pool(x)

        # Flatening

        x = torch.flatten(x, start_dim=1)

        # Fully connected Layers

        x = self.fc3(x)
        x = F.relu(x)
        x = self.dropout(x)

        # Output layer

        x = self.fc4(x)
        return x

In [7]:
# setting

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CustomCNN(num_classes=10).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=0.0001)

In [8]:
# Training Loop

num_epochs = 8

for epoch in range(num_epochs): # 10

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        # image12 0.1 -> 1 , .9 -> 2
        loss = criterion(outputs, labels)

        loss.backward()    ###
        optimizer.step()   ###

        _, predicted = torch.max(outputs, 1)
        # image12 -> 2 , image13 -> 8

        running_loss += loss.item() * labels.size(0)

        correct += (predicted == labels).sum().item()

        total += labels.size(0)

    avg_epoch_loss = running_loss / total
    epoch_acc = correct / total

    print(f"Epoch {epoch+1}/{num_epochs} | Loss: {avg_epoch_loss:.4f} | train acc: {epoch_acc:.4f}")

Epoch 1/8 | Loss: 0.7602 | train acc: 0.7744
Epoch 2/8 | Loss: 0.2073 | train acc: 0.9389
Epoch 3/8 | Loss: 0.1298 | train acc: 0.9614
Epoch 4/8 | Loss: 0.1010 | train acc: 0.9692
Epoch 5/8 | Loss: 0.0820 | train acc: 0.9748
Epoch 6/8 | Loss: 0.0732 | train acc: 0.9780
Epoch 7/8 | Loss: 0.0641 | train acc: 0.9810
Epoch 8/8 | Loss: 0.0590 | train acc: 0.9823


In [9]:
# Test Loop

model.eval()

test_loss= 0.0
correct = 0
total = 0
# 10,000 -> 20 * 500
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        test_loss += loss.item() * labels.size(0)

        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    avg_test_loss = test_loss / total
    test_acc = correct / total

    print(f"Test Loss: {avg_test_loss *100} | Test Acc: {test_acc * 100}")

Test Loss: 3.3550172392278905 | Test Acc: 98.78
